In [11]:
import os
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

MODEL_NAME = "gemini-3.5-flash"

QUANT_SYSTEM_PROMPT = """
You are an expert quantitative researcher building alphas on the WorldQuant BRAIN platform.
You think in terms of economic intuition, not data-mining.
- You ONLY use valid BRAIN FASTEXPR operators: rank, ts_zscore, ts_backfill, ts_mean, ts_delta, ts_std_dev, ts_decay_linear, reverse, add, subtract, hump, group_neutralize, winsorize.
- You ALWAYS use ts_backfill to handle sparse or delayed fundamental data.
- You respect whether a field is MATRIX or VECTOR.
- You avoid look-ahead bias.
You are concise and return well-structured output.
"""

print("Setup complete. Client ready.")

Setup complete. Client ready.


In [6]:
import sys
print(sys.executable)

c:\Users\HP 840G6 CORE i5\AppData\Local\Programs\Python\Python313\python.exe


In [14]:
def ask_llm(prompt, system_prompt=QUANT_SYSTEM_PROMPT):
    try:
        # Notice we are using 'client' (Gemini), NOT 'wq_client' (WorldQuant)
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt}
            ],
            temperature=0.2, # Low temperature for more analytical outputs
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {e}"

print("Helper function defined.")

Helper function defined.


In [3]:
# Dummy data representing fundamental17 dataset
data = {
    'id': ['fnd17_ebitda2ev_ttm', 'fnd17_ttmpr2rev', 'fnd17_aproipt', 'fnd17_1_2anfhsfcq', 'fnd17_revcngvr', 'fnd17_grosmgn'],
    'type': ['MATRIX', 'MATRIX', 'MATRIX', 'MATRIX', 'MATRIX', 'MATRIX'],
    'description': [
        'P/E excluding extraordinary items high, most recent quarter',
        'EBITDA yield captures cheap cash-generating businesses', 
        'High return on investment indicates efficient capital allocation',
        'Free cash flow per share signals real cash generation available to equity holders',
        'Year-over-year revenue growth captures improving demand and business momentum',
        'High gross margin reflects pricing power, cost control'
    ]
}
fields_df = pd.DataFrame(data)

def build_prompt_a(fields_df):
    fields_table = fields_df[['id', 'description']].to_markdown(index=False)
    prompt = f"""
    I will give you a table of WorldQuant BRAIN datafields.
    Dataset description: Direct Fundamental Data (USA/TOP3000)

    Your task: identify the 3 MOST PROMISING fields for alpha construction.
    Do NOT write any alpha expressions yet.

    For each chosen field return a row with:
    1. field name
    2. economic category (value / quality / growth / momentum)
    3. likely direction (is HIGH good or bad for future returns?)
    4. one-line reason it may predict returns
    5. suitable template families (level / change / stability / interaction)

    Rules:
    - Prefer economic intuition over data-mining.
    - Return a markdown table only.

    Datafields:
    {fields_table}
    """
    return prompt

prompt_a = build_prompt_a(fields_df)
print("Prompt A built. Ready to send.")

Prompt A built. Ready to send.


In [9]:
result = ask_llm(prompt_a)
print("\n--- LLM RESPONSE ---\n")
print(result)


--- LLM RESPONSE ---

| Field Name | Economic Category | Likely Direction | Reason | Suitable Template Families |
|:---|:---|:---|:---|:---|
| `fnd17_ebitda2ev_ttm` | Value | Positive (High is good) | Higher EBITDA-to-EV signals an undervalued company generating strong operating cash flows relative to its total enterprise value. | Level, Interaction, Change |
| `fnd17_aproipt` | Quality | Positive (High is good) | Superior return on invested capital reflects strong operational efficiency, economic moat, and effective capital allocation. | Level, Stability, Interaction |
| `fnd17_grosmgn` | Quality | Positive (High is good) | High gross margins indicate strong pricing power and structural competitive advantages that buffer against cost pressures. | Level, Change, Stability |


In [10]:
# These are the fields Gemini selected in the previous step
shortlisted_fields = ['fnd17_ebitda2ev_ttm', 'fnd17_aproipt', 'fnd17_grosmgn']

def build_prompt_b(fields_list):
    fields_str = "\n".join([f"- {f}" for f in fields_list])
    
    prompt = f"""
    For each datafield below, recommend the time-series parameters that FIT that field.
    Do NOT use a generic grid.

    For each field return JSON with keys:
    - "field": the field name
    - "update_frequency": your best guess (daily / monthly / quarterly / sparse)
    - "backfill_days": ONE recommended ts_backfill window (integer)
    - "lookbacks": a list of AT MOST 2 recommended lookback windows (integers)
    - "reason": one short sentence justifying the choice

    Guidance:
    - Quarterly/sparse fundamental fields -> large backfill (e.g. 60-120), longer lookbacks (e.g. 120, 250).
    - Faster/denser fields -> smaller backfill (5-20), shorter lookbacks (20-60).
    - Keep it minimal: 1 backfill and at most 2 lookbacks per field.

    Return a JSON array only, no prose. Format it exactly like this:
    [
      {{"field": "example_field", "update_frequency": "quarterly", "backfill_days": 120, "lookbacks": [120, 250], "reason": "..."}}
    ]

    Datafields:
    {fields_str}
    """
    return prompt

prompt_b = build_prompt_b(shortlisted_fields)
print("Prompt B built. Ready to send.")

Prompt B built. Ready to send.


In [11]:
import json
import re

def parse_json_array(text):
    """Extracts a JSON array from an LLM response, ignoring markdown formatting."""
    if not text:
        return []
    # Find the JSON array brackets
    match = re.search(r"\[.*\]", text, flags=re.DOTALL)
    if not match:
        return []
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return []

# Send the prompt to the LLM
print("Sending Prompt B to LLM...")
response_b = ask_llm(prompt_b)

# Parse the JSON response
param_recs = parse_json_array(response_b)

if param_recs:
    print("\n--- PARSED PARAMETERS ---")
    # Display nicely as a pandas DataFrame
    display(pd.DataFrame(param_recs))
else:
    print("\n--- ERROR PARSING JSON. RAW RESPONSE: ---\n")
    print(response_b)

Sending Prompt B to LLM...

--- PARSED PARAMETERS ---


,field,update_frequency,backfill_days,lookbacks,reason
0,fnd17_ebitda2ev_ttm,quarterly,120,"[120, 250]",EBITDA to EV relies on quarterly fundamental f...
1,fnd17_aproipt,quarterly,120,"[120, 250]",Income tax provision is updated in sparse quar...
2,fnd17_grosmgn,quarterly,120,"[120, 250]",Gross margin updates upon quarterly financial ...


In [12]:
# Valid operators to prevent hallucinations
COMMON_OPS = [
    "rank", "zscore", "normalize", "reverse", "ts_backfill", "ts_zscore", 
    "ts_mean", "ts_delta", "ts_std_dev", "ts_decay_linear", "ts_corr", 
    "ts_rank", "ts_sum", "add", "subtract", "hump"
]
ops_hint = ", ".join(COMMON_OPS)

def build_prompt_c(param_recs, ops_hint):
    prompt = f"""
    Generate 10 DIVERSE WorldQuant BRAIN alpha expression templates using fields from the dataset.

    Requirements:
    - Span DIFFERENT template families so the alphas are NOT correlated: level, change, smoothed, contrarian, stability, interaction, correlation, turnover-controlled.
    - Use ts_backfill on every field to handle sparse fundamentals.
    - Use the lookback/backfill values from the parameter recommendations below.
    - At most 5 unique operators per expression. Keep them interpretable.
    - For each, output: (a) the family, (b) the FASTEXPR expression, (c) one-line economic intuition, (d) whether high values should be long or short.

    Only use these operators: {ops_hint}

    Parameter recommendations (JSON):
    {json.dumps(param_recs, indent=2)}

    Return a markdown table with columns: family | expression | intuition | direction. Do not include any other text outside the markdown table.
    """
    return prompt

prompt_c = build_prompt_c(param_recs, ops_hint)
print("Prompt C built. Ready to send.")

Prompt C built. Ready to send.


In [13]:
print("Generating diverse alpha expressions...\n")
response_c = ask_llm(prompt_c)

print("--- GENERATED ALPHA TEMPLATES ---\n")
print(response_c)

Generating diverse alpha expressions...

--- GENERATED ALPHA TEMPLATES ---

| family | expression | intuition | direction |
| --- | --- | --- | --- |
| level | `rank(ts_backfill(fnd17_ebitda2ev_ttm, 120))` | High EBITDA to EV yield indicates undervaluation relative to peers. | Long |
| change | `rank(ts_delta(ts_backfill(fnd17_grosmgn, 120), 250))` | Year-over-year expansion in gross margin signals improving profitability. | Long |
| smoothed | `rank(ts_decay_linear(ts_backfill(fnd17_grosmgn, 120), 120))` | Linearly weighted recent gross margin captures persistent operational efficiency. | Long |
| contrarian | `reverse(ts_zscore(ts_backfill(fnd17_ebitda2ev_ttm, 120), 250))` | Overextended valuation yield spikes tend to mean-revert toward historical norms. | Short |
| stability | `reverse(rank(ts_std_dev(ts_backfill(fnd17_grosmgn, 120), 250)))` | Lower margin volatility reflects high earnings stability and lower business risk. | Long |
| interaction | `rank(add(ts_backfill(fnd17_ebitda

In [14]:
import re

# Extract the markdown table from the response
table_text = response_c

# Use regex to find all expressions enclosed in backticks ` `
expressions = re.findall(r'`([^`]+)`', table_text)

# Filter out any that accidentally grabbed non-expression text (like family names)
# We look for strings that contain an opening parenthesis '('
valid_expressions = [expr for expr in expressions if '(' in expr]

print(f"Extracted {len(valid_expressions)} expressions ready for simulation:\n")
for i, expr in enumerate(valid_expressions, 1):
    print(f"{i}. {expr}")

# Save this list for the next step
import json
with open('generated_alphas.json', 'w') as f:
    json.dump(valid_expressions, f, indent=4)

Extracted 10 expressions ready for simulation:

1. rank(ts_backfill(fnd17_ebitda2ev_ttm, 120))
2. rank(ts_delta(ts_backfill(fnd17_grosmgn, 120), 250))
3. rank(ts_decay_linear(ts_backfill(fnd17_grosmgn, 120), 120))
4. reverse(ts_zscore(ts_backfill(fnd17_ebitda2ev_ttm, 120), 250))
5. reverse(rank(ts_std_dev(ts_backfill(fnd17_grosmgn, 120), 250)))
6. rank(add(ts_backfill(fnd17_ebitda2ev_ttm, 120), ts_backfill(fnd17_grosmgn, 120)))
7. ts_corr(ts_backfill(fnd17_ebitda2ev_ttm, 120), ts_backfill(fnd17_grosmgn, 120), 250)
8. hump(rank(ts_delta(ts_backfill(fnd17_aproipt, 120), 120)), 0.05)
9. ts_rank(ts_backfill(fnd17_ebitda2ev_ttm, 120), 250)
10. ts_zscore(ts_delta(ts_backfill(fnd17_grosmgn, 120), 120), 250)


In [16]:
import requests
import time
import os
import json

# 1. Authenticate with BRAIN
def start_brain_session():
    s = requests.Session()
    # BRAIN API expects specific headers for authentication
    s.auth = (os.getenv("BRAIN_EMAIL"), os.getenv("BRAIN_PASSWORD"))
    
    # Trigger the authentication
    resp = s.post("https://api.worldquantbrain.com/authentication")
    
    if resp.status_code == 201:
        print("✅ Successfully logged into WorldQuant BRAIN!")
        return s
    else:
        print(f"❌ Login failed. Status code: {resp.status_code}")
        print(resp.text)
        return None

# Start the session
brain_session = start_brain_session()

# 2. Define the Simulation Function
def simulate_alpha(session, expression):
    payload = {
        "type": "REGULAR",
        "settings": {
            "instrumentType": "EQUITY",
            "region": "USA",
            "universe": "TOP3000",
            "delay": 1,
            "decay": 0,
            "neutralization": "SUBINDUSTRY", # Standard baseline
            "truncation": 0.08,
            "pasteurization": "ON",
            "unitHandling": "VERIFY",
            "nanHandling": "OFF",
            "language": "FASTEXPR",
            "visualization": False,
        },
        "regular": expression,
    }
    
    # Send the simulation request
    resp = session.post("https://api.worldquantbrain.com/simulations", json=payload)
    
    if resp.status_code != 201:
        print(f"Error submitting expression: {expression}")
        print(resp.text)
        return None
        
    # Get the simulation ID from the response headers
    sim_url = resp.headers["Location"]
    
    print(f"⏳ Simulating: {expression[:50]}...")
    
    # Poll the API until the simulation is done
    while True:
        status_resp = session.get(sim_url)
        if status_resp.status_code == 200:
            data = status_resp.json()
            if data.get("status") == "COMPLETE":
                # Extract the metrics
                alpha_id = data.get("alpha")
                metrics = session.get(f"https://api.worldquantbrain.com/alphas/{alpha_id}").json()
                return metrics
            elif data.get("status") == "ERROR":
                print(f"❌ Simulation failed for: {expression}")
                print(data.get("message"))
                return None
        time.sleep(3) # Be polite to the API, wait 3 seconds before checking again

❌ Login failed. Status code: 401
{"detail":"INVALID_CREDENTIALS"}


In [17]:
from brain_login_and_sim import BrainClient

def start_brain_session():
    # This will handle email/password AND any biometric scan requirements
    client = BrainClient().authenticate()
    print("✅ Successfully logged into BRAIN via BrainClient!")
    return client

In [19]:
from brain_login_and_sim import BrainClient, Simulator

# Authenticate (the BrainClient handles the biometric scan if needed)
client = BrainClient().authenticate()
print("✅ Successfully logged into BRAIN via BrainClient!")

# Create the Simulator object
sim = Simulator(client)
print("✅ Simulator is ready.")

BrainAuthError: No credentials found - set WQ_EMAIL/WQ_PASSWORD in a .env file (see .env.example), or pass email/password to BrainClient, or use a JSON file via WQ_CREDENTIALS_FILE. If set in .env but not found, make sure python-dotenv is installed and .env is next to your code.

In [23]:
import os
from dotenv import load_dotenv
from brain_login_and_sim import BrainClient, Simulator

# Force dotenv to look in the exact folder this notebook is running from
notebook_dir = os.getcwd()
env_path = os.path.join(notebook_dir, '.env')
load_dotenv(dotenv_path=env_path)

# Debugging: Print to check if the variables are loaded (don't print the password!)
email = os.getenv("WQ_EMAIL")
print(f"Loaded email: {email}")

# If the .env is still failing, we can pass the credentials directly as a fallback
try:
    client = BrainClient().authenticate()
except Exception as e:
    print(f"Env failed, trying direct passing... Error: {e}")
    # Fallback: pass directly (replace with your actual email/password if env fails)
    client = BrainClient(email=email, password=os.getenv("WQ_PASSWORD")).authenticate()

print("✅ Successfully logged into BRAIN via BrainClient!")

# Create the Simulator object
sim = Simulator(client)
print("✅ Simulator is ready.")

Loaded email: your_worldquant_karanjajoseph1856@gmail.com
Env failed, trying direct passing... Error: Login failed (401) - check email/password. response: {"detail":"INVALID_CREDENTIALS"}


BrainAuthError: Login failed (401) - check email/password. response: {"detail":"INVALID_CREDENTIALS"}

In [26]:
from brain_login_and_sim import BrainClient, Simulator

# Type your exact credentials inside the quotes below
test_email =karanjajoseph1856@gmail.com
test_password =CartoonDuck@14

try:
    print("Testing direct login...")
    # We pass the strings directly to the client, no .env needed
    client = BrainClient(email=test_email, password=test_password).authenticate()
    print("✅ SUCCESS! The credentials are correct.")
    
    # Create the Simulator
    sim = Simulator(client)
    print("✅ Simulator is ready.")
    
except Exception as e:
    print("❌ Login Failed. The issue is the credentials, not the code.")
    print(f"Error: {e}")

NameError: name 'karanjajoseph1856' is not defined

In [2]:
from brain_login_and_sim import BrainClient, Simulator

test_email = "karanjajoseph1856@gmail.com"
test_password = "CartoonDuck@14" # Use your new password!

try:
    print("Testing direct login...")
    # RENAME TO wq_client HERE
    wq_client = BrainClient(email=test_email, password=test_password).authenticate()
    print("✅ SUCCESS! The credentials are correct.")
    
    # RENAME TO wq_client HERE
    sim = Simulator(wq_client)
    print("✅ Simulator is ready.")
    
except Exception as e:
    print("❌ Login Failed.")
    print(f"Error: {e}")

Testing direct login...
✅ SUCCESS! The credentials are correct.
✅ Simulator is ready.


In [28]:
import time
import pandas as pd

results = []

# Check if valid_expressions exists from your earlier cell
if 'valid_expressions' not in locals():
    print("Error: valid_expressions not found. Please re-run the cell that extracts them from the LLM response.")
else:
    print(f"Starting simulation of {len(valid_expressions)} alphas...\n")
    
    for expr in valid_expressions:
        print(f"⏳ Simulating: {expr[:60]}...")
        try:
            # Use the Simulator object we just created
            r = sim.simulate(expr)
            
            # Check the simulation status
            status = r.get("simulation", {}).get("status")
            if status == "COMPLETE":
                # Extract metrics
                is_stats = r.get("alpha", {}).get("is", {})
                sharpe = is_stats.get("sharpe", 0)
                fitness = is_stats.get("fitness", 0)
                turnover = is_stats.get("turnover", 0)
                returns = is_stats.get("returns", 0)
                
                print(f"✅ Done! Sharpe: {sharpe:.2f} | Fitness: {fitness:.2f} | Turnover: {turnover:.2f}")
                
                results.append({
                    "expression": expr,
                    "sharpe": sharpe,
                    "fitness": fitness,
                    "turnover": turnover,
                    "returns": returns
                })
                
                # Check if it's a high-quality alpha
                if abs(fitness) >= 1.0 and abs(sharpe) >= 1.25:
                    print("🌟 EXCELLENT ALPHA! Candidate for submission.")
            else:
                print(f"❌ Simulation failed. Status: {status}")
                # Uncomment the line below if you want to see the full error message
                # print(r)
                
        except Exception as e:
            print(f"❌ Error simulating {expr}: {e}")
        
        print("-" * 50)
        time.sleep(1) # Short pause between simulations to be safe

    # Display the final results table
    if results:
        print("\n--- FINAL RESULTS ---")
        df_results = pd.DataFrame(results).sort_values(by="fitness", ascending=False)
        display(df_results)
    else:
        print("\nNo simulations completed successfully.")

Starting simulation of 10 alphas...

⏳ Simulating: rank(ts_backfill(fnd17_ebitda2ev_ttm, 120))...
[17:58:42] simulating: rank(ts_backfill(fnd17_ebitda2ev_ttm, 120))
❌ Simulation failed. Status: ERROR
--------------------------------------------------
⏳ Simulating: rank(ts_delta(ts_backfill(fnd17_grosmgn, 120), 250))...
[17:59:24] simulating: rank(ts_delta(ts_backfill(fnd17_grosmgn, 120), 250))
❌ Simulation failed. Status: ERROR
--------------------------------------------------
⏳ Simulating: rank(ts_decay_linear(ts_backfill(fnd17_grosmgn, 120), 120))...
[17:59:42] simulating: rank(ts_decay_linear(ts_backfill(fnd17_grosmgn, 120), 120))
❌ Simulation failed. Status: ERROR
--------------------------------------------------
⏳ Simulating: reverse(ts_zscore(ts_backfill(fnd17_ebitda2ev_ttm, 120), 250...
[17:59:55] simulating: reverse(ts_zscore(ts_backfill(fnd17_ebitda2ev_ttm, 120), 250))
❌ Simulation failed. Status: ERROR
--------------------------------------------------
⏳ Simulating: reverse

In [29]:
            else:
                print(f"❌ Simulation failed. Status: {status}")
                # Print the full response to see the specific error message
                print("Full API Response:")
                print(r) 

SyntaxError: invalid syntax (1845420759.py, line 1)

In [30]:
# Test a single, basic expression
test_expr = "rank(fnd17_ebitda2ev_ttm)"

print(f"Testing expression: {test_expr}")
try:
    r = sim.simulate(test_expr)
    print("Full API Response:")
    print(r)
except Exception as e:
    print(f"Error: {e}")

Testing expression: rank(fnd17_ebitda2ev_ttm)
[18:03:06] simulating: rank(fnd17_ebitda2ev_ttm)
Full API Response:
{'simulation': {'id': '4hMP9364F4re9hDl6Wl9bFy', 'type': 'REGULAR', 'status': 'ERROR', 'message': 'Invalid data field fnd17_ebitda2ev_ttm. <linkToCommonErrorMessages>Learn more</linkToCommonErrorMessages>', 'location': {'line': None, 'start': None, 'end': None, 'property': 'regular'}, 'links': {'linkToCommonErrorMessages': 'https://support.worldquantbrain.com/hc/en-us/articles/18423410021783--BRAIN-TIPS-Troubleshooting-common-error-messages'}}, 'alpha_id': None, 'alpha': None, 'promising': False, 'saved': False}


In [31]:
import requests
import pandas as pd

# Use your authenticated session (brain_session from Cell 10)
# If using the wqb library, you can use wqbs.search_operators() instead[reference:1]
operators_resp = brain_session.get("https://api.worldquantbrain.com/operators")
operators_df = pd.DataFrame(operators_resp.json())

# Display the columns to see what metadata is available
print(operators_df.columns.tolist())
print(f"Fetched {len(operators_df)} operators.")

AttributeError: 'NoneType' object has no attribute 'get'

In [33]:
import requests

# Fetch the datafields again
fields_resp = client.session.get(
    "https://api.worldquantbrain.com/data-fields",
    params={"region": "USA", "universe": "TOP3000", "delay": 1, "dataset.id": "fundamental17"}
)

print(f"Status Code: {fields_resp.status_code}")
print("Response Preview:")
# Print the first 500 characters of the response to see what went wrong
print(str(fields_resp.json())[:500])

Status Code: 400
Response Preview:
['Invalid query']


In [34]:
import pandas as pd

active_session = client.session

# 1. Fetch the list of available datasets
print("Fetching available datasets...")
datasets_resp = active_session.get(
    "https://api.worldquantbrain.com/data-sets",
    params={"region": "USA", "universe": "TOP3000", "delay": 1, "limit": 20} 
)
datasets_data = datasets_resp.json()

if 'results' in datasets_data and len(datasets_data['results']) > 0:
    # Let's pick a fundamental or analyst dataset if available
    target_ds = datasets_data['results'][0] 
    for ds in datasets_data['results']:
        if 'fundamental' in ds['id'].lower() or 'analyst' in ds['id'].lower():
            target_ds = ds
            break
            
    dataset_id = target_ds['id']
    print(f"✅ Found dataset: {dataset_id} ({target_ds['name']})")
    
    # 2. Fetch the fields for this dataset
    fields_resp = active_session.get(
        "https://api.worldquantbrain.com/data-fields",
        params={"region": "USA", "universe": "TOP3000", "delay": 1, "dataset.id": dataset_id, "limit": 40}
    )
    fields_data = fields_resp.json()
    
    if 'results' in fields_data and len(fields_data['results']) > 0:
        fields_df = pd.DataFrame(fields_data['results'])
        fields_df = fields_df[['id', 'description']]
        fields_table_md = fields_df.to_markdown(index=False)
        print(f"✅ Fetched {len(fields_df)} valid datafields.")
    else:
        print("❌ No fields found.")
else:
    print("❌ Could not fetch datasets.")

Fetching available datasets...
✅ Found dataset: analyst4 (Analyst Estimate Data for Equity)
❌ No fields found.


In [6]:
import pandas as pd
import json

active_session = wq_client.session

# 1. Fetch available datasets
print("Fetching available datasets...")
datasets_resp = active_session.get(
    "https://api.worldquantbrain.com/data-sets",
    params={"region": "USA", "universe": "TOP3000", "delay": 1, "instrumentType": "EQUITY", "limit": 20} 
)
datasets_data = datasets_resp.json()

if 'results' in datasets_data and len(datasets_data['results']) > 0:
    # Let's try to find a fundamental dataset first, if not, grab the first one
    target_ds = datasets_data['results'][0] 
    for ds in datasets_data['results']:
        if 'fundamental' in ds['id'].lower():
            target_ds = ds
            break
            
    dataset_id = target_ds['id']
    print(f"✅ Found dataset: {dataset_id} ({target_ds['name']})")
    
    # 2. Fetch the fields (Adding instrumentType is CRITICAL)
    print(f"Fetching fields for {dataset_id}...")
    fields_resp = active_session.get(
        "https://api.worldquantbrain.com/data-fields",
        params={
            "region": "USA", 
            "universe": "TOP3000", 
            "delay": 1, 
            "dataset.id": dataset_id, 
            "instrumentType": "EQUITY", # <--- This was likely missing
            "limit": 40
        }
    )
    fields_data = fields_resp.json()
    
    if 'results' in fields_data and len(fields_data['results']) > 0:
        fields_df = pd.DataFrame(fields_data['results'])
        fields_df = fields_df[['id', 'description']]
        fields_table_md = fields_df.to_markdown(index=False)
        print(f"✅ Fetched {len(fields_df)} valid datafields.")
        display(fields_df.head())
    else:
        print(f"❌ No fields found for {dataset_id}.")
        print("Raw API Response:")
        print(json.dumps(fields_data, indent=2)) # This will show us the exact error
else:
    print("❌ Could not fetch datasets. API Response:")
    print(json.dumps(datasets_data, indent=2))

Fetching available datasets...
✅ Found dataset: fundamental2 (Report Footnotes)
Fetching fields for fundamental2...
✅ Fetched 40 valid datafields.


,id,description
0,accrued_liabilities_total,Total amount of accrued liabilities.
1,accrued_liabilities_total_2,Total amount of liabilities that have been inc...
2,accumulated_amortization_customer_intangibles,Total accumulated amortization for customer-re...
3,accumulated_amortization_customer_intangibles_2,Total amortization recorded for customer-relat...
4,accumulated_amortization_finite_intangibles,Total amortization recorded for finite-lived i...


In [36]:
# Valid operators strictly from your provided text file
VALID_OPERATORS = [
    "abs", "add", "densify", "divide", "inverse", "log", "max", "min", "multiply", "power", "reverse", "sign", "signed_power", "sqrt", "subtract",
    "and", "if_else", "is_nan", "not", "or",
    "days_from_last_change", "hump", "kth_element", "last_diff_value", "ts_arg_max", "ts_arg_min", "ts_av_diff", "ts_backfill", "ts_corr", "ts_count_nans", "ts_covariance", "ts_decay_linear", "ts_delay", "ts_delta", "ts_mean", "ts_product", "ts_quantile", "ts_rank", "ts_regression", "ts_scale", "ts_std_dev", "ts_step", "ts_sum", "ts_zscore",
    "normalize", "quantile", "rank", "scale", "winsorize", "zscore",
    "vec_avg", "vec_sum", "bucket", "trade_when",
    "group_backfill", "group_mean", "group_neutralize", "group_rank", "group_scale", "group_zscore"
]
ops_hint = ", ".join(VALID_OPERATORS)

In [37]:
prompt_c = f"""
    Generate 5 DIVERSE WorldQuant BRAIN alpha expression templates using the fields provided below.
    
    CRITICAL RULES:
    1. You MUST ONLY use the following operators: {ops_hint}
    2. Do NOT invent new operators. 
    3. Span DIFFERENT template families (level, change, smoothed, contrarian, stability).
    4. Use ts_backfill(x, 120) on the fields to handle sparse fundamental data.
    5. For each, output: (a) the family, (b) the FASTEXPR expression, (c) one-line economic intuition.

    Here are the available fields:
    {fields_table_md}

    Return a markdown table with columns: family | expression | intuition.
"""

print("Generating strictly valid alpha expressions...\n")
response_c = ask_llm(prompt_c)
print(response_c)

Generating strictly valid alpha expressions...

Error: 'BrainClient' object has no attribute 'chat'


In [39]:
from brain_login_and_sim import BrainClient, Simulator

# Rename the variable to wq_client
wq_client = BrainClient(email="karanjajoseph1856@gmail.com", password="CartoonDuck@14").authenticate()
sim = Simulator(wq_client)

# Also update any cells that use client.session to wq_client.session
active_session = wq_client.session

In [5]:
# Re-define the valid operators from your text file
VALID_OPERATORS = [
    "abs", "add", "densify", "divide", "inverse", "log", "max", "min", "multiply", "power", "reverse", "sign", "signed_power", "sqrt", "subtract",
    "and", "if_else", "is_nan", "not", "or",
    "days_from_last_change", "hump", "kth_element", "last_diff_value", "ts_arg_max", "ts_arg_min", "ts_av_diff", "ts_backfill", "ts_corr", "ts_count_nans", "ts_covariance", "ts_decay_linear", "ts_delay", "ts_delta", "ts_mean", "ts_product", "ts_quantile", "ts_rank", "ts_regression", "ts_scale", "ts_std_dev", "ts_step", "ts_sum", "ts_zscore",
    "normalize", "quantile", "rank", "scale", "winsorize", "zscore",
    "vec_avg", "vec_sum", "bucket", "trade_when",
    "group_backfill", "group_mean", "group_neutralize", "group_rank", "group_scale", "group_zscore"
]

ops_hint = ", ".join(VALID_OPERATORS)
print("✅ Operators loaded into memory.")

✅ Operators loaded into memory.


In [19]:
prompt_c = f"""
    Generate 5 DIVERSE WorldQuant BRAIN alpha expression templates using the fields provided below.
    
    CRITICAL RULES:
    1. You MUST ONLY use the following operators: {ops_hint}
    2. Do NOT invent new operators. 
    3. Span DIFFERENT template families (level, change, smoothed, contrarian, stability).
    4. Use ts_backfill(x, 120) on the fields to handle sparse fundamental data.
    5. For each, output: (a) the family, (b) the FASTEXPR expression, (c) one-line economic intuition.

    Here are the available fields:
    {fields_table_md}

    Return a markdown table with columns: family | expression | intuition.
"""

print("Generating strictly valid alpha expressions...\n")
response_c = ask_llm(prompt_c)
print(response_c)

Generating strictly valid alpha expressions...

| family | expression | intuition |
| :--- | :--- | :--- |
| **Level** | `group_neutralize(rank(divide(ts_backfill(assets_fair_value_lvl1, 120), add(ts_backfill(assets_fair_value_lvl1, 120), ts_backfill(assets_fair_value_lvl2, 120)))), subindustry)` | Higher proportion of Level 1 mark-to-market assets relative to opaque Level 2 assets indicates superior valuation transparency and asset quality. |
| **Change** | `reverse(group_neutralize(ts_zscore(ts_backfill(allocated_sbp_expense_total, 120), 252), industry))` | Sharp acceleration in share-based compensation expenses signals potential shareholder dilution and aggressive management pay structures. |
| **Smoothed** | `group_neutralize(rank(ts_decay_linear(divide(ts_backfill(acquired_goodwill_value, 120), add(ts_backfill(acquisition_related_costs_expense, 120), 1)), 60)), subindustry)` | Exponentially weighted trend of goodwill created per dollar of M&A transaction expense measures inorganic

In [20]:
import re
import time
import pandas as pd

# 1. Extract expressions from the markdown table
# This regex looks for text wrapped in backticks ` `
expressions = re.findall(r'`([^`]+)`', response_c)
# Filter to ensure we only get valid FASTEXPR strings (must contain a parenthesis)
valid_expressions = [expr for expr in expressions if '(' in expr]

print(f"Extracted {len(valid_expressions)} expressions. Starting simulations...\n")

results = []

# 2. Simulate each expression
for expr in valid_expressions:
    print(f"⏳ Simulating: {expr[:50]}...")
    try:
        # Send to BRAIN
        r = sim.simulate(expr)
        status = r.get("simulation", {}).get("status")
        
        if status == "COMPLETE":
            # Extract the performance metrics
            is_stats = r.get("alpha", {}).get("is", {})
            sharpe = is_stats.get("sharpe", 0)
            fitness = is_stats.get("fitness", 0)
            turnover = is_stats.get("turnover", 0)
            
            print(f"✅ Done! Sharpe: {sharpe:.2f} | Fitness: {fitness:.2f} | Turnover: {turnover:.2f}")
            
            results.append({
                "expression": expr, 
                "sharpe": sharpe, 
                "fitness": fitness, 
                "turnover": turnover
            })
            
            # Highlight good alphas
            if abs(fitness) >= 1.0 and abs(sharpe) >= 1.25:
                print("🌟 EXCELLENT ALPHA! Candidate for submission.")
        else:
            print(f"❌ Simulation failed. Status: {status}")
            # Print the exact error message from the API
            error_msg = r.get("simulation", {}).get("message", "Unknown error")
            print(f"API Message: {error_msg}")
            
    except Exception as e:
        print(f"❌ Error connecting to API: {e}")
        
    print("-" * 50)
    time.sleep(1) # Be polite to the server

# 3. Display the final leaderboard
if results:
    print("\n--- FINAL SIMULATION RESULTS ---")
    df_results = pd.DataFrame(results).sort_values(by="fitness", ascending=False)
    display(df_results)
else:
    print("\nNo simulations completed successfully. Check the API messages above.")

Extracted 5 expressions. Starting simulations...

⏳ Simulating: group_neutralize(rank(divide(ts_backfill(assets_fa...
[18:56:29] simulating: group_neutralize(rank(divide(ts_backfill(assets_fair_value_lvl1, 120),
✅ Done! Sharpe: -0.90 | Fitness: -0.45 | Turnover: 0.01
--------------------------------------------------
⏳ Simulating: reverse(group_neutralize(ts_zscore(ts_backfill(all...
[18:58:01] simulating: reverse(group_neutralize(ts_zscore(ts_backfill(allocated_sbp_expense_t
✅ Done! Sharpe: 0.76 | Fitness: 0.36 | Turnover: 0.09
--------------------------------------------------
⏳ Simulating: group_neutralize(rank(ts_decay_linear(divide(ts_ba...
[18:59:28] simulating: group_neutralize(rank(ts_decay_linear(divide(ts_backfill(acquired_good
✅ Done! Sharpe: -0.13 | Fitness: -0.02 | Turnover: 0.01
--------------------------------------------------
⏳ Simulating: reverse(group_neutralize(rank(ts_delta(ts_backfill...
[19:01:08] simulating: reverse(group_neutralize(rank(ts_delta(ts_backfill(all

,expression,sharpe,fitness,turnover
1,reverse(group_neutralize(ts_zscore(ts_backfill...,0.76,0.36,0.0857
4,reverse(group_neutralize(rank(divide(ts_std_de...,0.33,0.12,0.0202
3,reverse(group_neutralize(rank(ts_delta(ts_back...,0.04,0.01,0.0648
2,group_neutralize(rank(ts_decay_linear(divide(t...,-0.13,-0.02,0.0104
0,group_neutralize(rank(divide(ts_backfill(asset...,-0.90,-0.45,0.0110


In [21]:
import time
import pandas as pd

# Let's create mutated versions of your best expressions
mutations = [
    # 1. Reverse the worst one (Row 0) to see if it hits +0.90
    "reverse(group_neutralize(rank(divide(ts_backfill(assets_fair_value_lvl1, 120), add(ts_backfill(assets_fair_value_lvl1, 120), ts_backfill(assets_fair_value_lvl2, 120)))), subindustry))",
    
    # 2. Improve the best one (Row 1) by blending it with a 5-day price momentum
    # This increases turnover and adds a technical edge
    "rank(add(reverse(group_neutralize(ts_zscore(ts_backfill(allocated_sbp_expense_total, 120), 252), industry)), ts_delta(close, 5)))",
    
    # 3. Best one but with a shorter lookback (60 days instead of 252) to increase turnover
    "reverse(group_neutralize(ts_zscore(ts_backfill(allocated_sbp_expense_total, 120), 60), industry))"
]

results_mutated = []

for expr in mutations:
    print(f"⏳ Simulating Mutation: {expr[:60]}...")
    try:
        r = sim.simulate(expr)
        status = r.get("simulation", {}).get("status")
        
        if status == "COMPLETE":
            is_stats = r.get("alpha", {}).get("is", {})
            sharpe = is_stats.get("sharpe", 0)
            fitness = is_stats.get("fitness", 0)
            turnover = is_stats.get("turnover", 0)
            
            print(f"✅ Done! Sharpe: {sharpe:.2f} | Fitness: {fitness:.2f} | Turnover: {turnover:.2f}")
            results_mutated.append({"expression": expr, "sharpe": sharpe, "fitness": fitness, "turnover": turnover})
        else:
            print(f"❌ Simulation failed.")
            
    except Exception as e:
        print(f"❌ Error: {e}")
    print("-" * 50)
    time.sleep(1)

if results_mutated:
    display(pd.DataFrame(results_mutated).sort_values(by="fitness", ascending=False))

⏳ Simulating Mutation: reverse(group_neutralize(rank(divide(ts_backfill(assets_fair...
[19:07:52] simulating: reverse(group_neutralize(rank(divide(ts_backfill(assets_fair_value_lvl
✅ Done! Sharpe: 0.90 | Fitness: 0.45 | Turnover: 0.01
--------------------------------------------------
⏳ Simulating Mutation: rank(add(reverse(group_neutralize(ts_zscore(ts_backfill(allo...
[19:09:51] simulating: rank(add(reverse(group_neutralize(ts_zscore(ts_backfill(allocated_sbp_
❌ Simulation failed.
--------------------------------------------------
⏳ Simulating Mutation: reverse(group_neutralize(ts_zscore(ts_backfill(allocated_sbp...
[19:11:23] simulating: reverse(group_neutralize(ts_zscore(ts_backfill(allocated_sbp_expense_t
✅ Done! Sharpe: 0.55 | Fitness: 0.18 | Turnover: 0.26
--------------------------------------------------


,expression,sharpe,fitness,turnover
0,reverse(group_neutralize(rank(divide(ts_backfi...,0.90,0.45,0.011
1,reverse(group_neutralize(ts_zscore(ts_backfill...,0.55,0.18,0.258


In [3]:
import pandas as pd
import json

# 1. Fetch available datasets for TOP1000
print("Fetching datasets for TOP1000...")
datasets_resp = wq_client.session.get(
    "https://api.worldquantbrain.com/data-sets",
    params={"region": "USA", "universe": "TOP1000", "delay": 1, "instrumentType": "EQUITY", "limit": 50} 
)
datasets_data = datasets_resp.json()

if 'results' in datasets_data and len(datasets_data['results']) > 0:
    # Find an analyst dataset
    target_ds = None
    for ds in datasets_data['results']:
        if 'analyst' in ds['id'].lower() or 'analyst' in ds.get('name', '').lower():
            target_ds = ds
            break
            
    if not target_ds:
        target_ds = datasets_data['results'][0] # Fallback
        
    dataset_id = target_ds['id']
    print(f"✅ Found dataset: {dataset_id} ({target_ds['name']})")
    
    # 2. Fetch the fields
    fields_resp = wq_client.session.get(
        "https://api.worldquantbrain.com/data-fields",
        params={
            "region": "USA", 
            "universe": "TOP1000", 
            "delay": 1, 
            "dataset.id": dataset_id, 
            "instrumentType": "EQUITY",
            "limit": 40
        }
    )
    fields_data = fields_resp.json()
    
    if 'results' in fields_data and len(fields_data['results']) > 0:
        fields_df = pd.DataFrame(fields_data['results'])
        # Keep only id and description
        fields_df = fields_df[['id', 'description']]
        fields_table_md = fields_df.to_markdown(index=False)
        print(f"✅ Fetched {len(fields_df)} valid datafields.")
        display(fields_df.head())
    else:
        print("❌ No fields found.")
        print(fields_data)
else:
    print("❌ Could not fetch datasets.")

Fetching datasets for TOP1000...
✅ Found dataset: analyst4 (Analyst Estimate Data for Equity)
✅ Fetched 40 valid datafields.


,id,description
0,actual_cashflow_per_share_value_quarterly,Cash Flow Per Share - actual value for the qua...
1,actual_dividend_value_quarterly,Dividend - Actual value for the quarter
2,actual_eps_value_quarterly,Earnings Per Share (Income Statement/Per Share...
3,actual_sales_value_annual,Sales - Actual Value
4,actual_sales_value_quarterly,Sales - Value in financial services income sta...


In [15]:
# Assuming ops_hint is still in memory. If not, re-run your operators cell.
prompt_c_analyst = f"""
    Generate 5 DIVERSE WorldQuant BRAIN alpha expression templates using Analyst Estimate fields.
    
    CRITICAL RULES:
    1. You MUST ONLY use the following operators: {ops_hint}
    2. Do NOT invent new operators. 
    3. Span DIFFERENT template families (level, change, smoothed, contrarian, stability).
    4. Analysts constantly revise their estimates. Use shorter backfills (e.g., ts_backfill(x, 5)) and shorter lookbacks (e.g., 20, 60) to capture these revisions.
    5. For each, output: (a) the family, (b) the FASTEXPR expression, (c) one-line economic intuition.

    Here are the available fields:
    {fields_table_md}

    Return a markdown table with columns: family | expression | intuition.
"""

print("Generating Analyst alpha expressions...\n")
response_c_analyst = ask_llm(prompt_c_analyst)
print(response_c_analyst)

Generating Analyst alpha expressions...

| family | expression | intuition |
| :--- | :--- | :--- |
| **Level** | `group_neutralize(rank(ts_zscore(ts_backfill(anl4_adxqfv110_mean, 5), 60)), subindustry)` | High current quarterly analyst consensus mean relative to its recent 60-day trend signals positive short-term fundamental momentum. |
| **Change** | `group_neutralize(ts_delta(ts_backfill(subtract(anl4_adxqfv110_pu, anl4_adxqfv110_down), 5), 20), subindustry)` | Acceleration in net analyst upgrades over the past 20 trading days captures institutional estimate revision momentum. |
| **Smoothed** | `rank(ts_decay_linear(ts_delta(ts_backfill(anl4_ady_mean, 5), 20), 10))` | Linearly decayed recent upgrades in annual mean consensus filter out single-day noise and isolate persistent earnings revisions. |
| **Contrarian** | `reverse(group_neutralize(ts_zscore(ts_backfill(anl4_adxqfv110_mean, 5), 20), subindustry))` | Fades temporary overreactions caused by aggressive short-term estimate spi

In [16]:
import re
import time

# Extract expressions
expressions = re.findall(r'`([^`]+)`', response_c_analyst)
valid_expressions = [expr for expr in expressions if '(' in expr]

print(f"Extracted {len(valid_expressions)} expressions. Simulating on TOP1000...\n")

results_top1000 = []

for expr in valid_expressions:
    print(f"⏳ Simulating: {expr[:60]}...")
    try:
        # HERE IS THE MAGIC: We pass settings to override the default universe
        r = sim.simulate(
            expr, 
            settings={
                "region": "USA",
                "universe": "TOP1000", # <--- TARGETING TOP1000
                "delay": 1,
                "decay": 0,
                "neutralization": "SUBINDUSTRY"
            }
        )
        
        status = r.get("simulation", {}).get("status")
        
        if status == "COMPLETE":
            is_stats = r.get("alpha", {}).get("is", {})
            sharpe = is_stats.get("sharpe", 0)
            fitness = is_stats.get("fitness", 0)
            turnover = is_stats.get("turnover", 0)
            
            print(f"✅ Done! Sharpe: {sharpe:.2f} | Fitness: {fitness:.2f} | Turnover: {turnover:.2f}")
            results_top1000.append({"expression": expr, "sharpe": sharpe, "fitness": fitness, "turnover": turnover})
        else:
            print(f"❌ Simulation failed. Status: {status}")
            print(r.get("simulation", {}).get("message", ""))
            
    except Exception as e:
        print(f"❌ Error: {e}")
    print("-" * 50)
    time.sleep(1)

if results_top1000:
    print("\n--- FINAL RESULTS (TOP1000 ANALYST DATA) ---")
    display(pd.DataFrame(results_top1000).sort_values(by="fitness", ascending=False))

Extracted 5 expressions. Simulating on TOP1000...

⏳ Simulating: group_neutralize(rank(ts_zscore(ts_backfill(anl4_adxqfv110_m...
[06:58:13] simulating: group_neutralize(rank(ts_zscore(ts_backfill(anl4_adxqfv110_mean, 5), 6
❌ Error: 400 Client Error: Bad Request for url: https://api.worldquantbrain.com/simulations
--------------------------------------------------
⏳ Simulating: group_neutralize(ts_delta(ts_backfill(subtract(anl4_adxqfv11...
[06:58:16] simulating: group_neutralize(ts_delta(ts_backfill(subtract(anl4_adxqfv110_pu, anl4
❌ Error: 400 Client Error: Bad Request for url: https://api.worldquantbrain.com/simulations
--------------------------------------------------
⏳ Simulating: rank(ts_decay_linear(ts_delta(ts_backfill(anl4_ady_mean, 5),...
[06:58:17] simulating: rank(ts_decay_linear(ts_delta(ts_backfill(anl4_ady_mean, 5), 20), 10))
❌ Error: 400 Client Error: Bad Request for url: https://api.worldquantbrain.com/simulations
--------------------------------------------------
⏳ Si

In [21]:
import re
import time
import pandas as pd

# 1. Extract the expressions from the LLM's Analyst response
# (Make sure 'response_c_analyst' is still in memory. If not, re-run the Analyst prompt cell)
expressions = re.findall(r'`([^`]+)`', response_c_analyst)
valid_expressions = [expr for expr in expressions if '(' in expr]

print(f"Extracted {len(valid_expressions)} analyst expressions. Simulating on TOP1000...\n")

results_top1000 = []

# 2. Use the full_settings dictionary we just built
for expr in valid_expressions:
    print(f"⏳ Simulating: {expr[:60]}...")
    try:
        r = sim.simulate(expr, settings=full_settings)
        
        # Robust error checking
        alpha_data = r.get("alpha")
        sim_status = r.get("simulation", {}).get("status")
        
        if alpha_data and sim_status == "COMPLETE":
            is_stats = alpha_data.get("is", {})
            sharpe = is_stats.get("sharpe", 0)
            fitness = is_stats.get("fitness", 0)
            turnover = is_stats.get("turnover", 0)
            
            print(f"✅ Done! Sharpe: {sharpe:.2f} | Fitness: {fitness:.2f} | Turnover: {turnover:.2f}")
            results_top1000.append({"expression": expr, "sharpe": sharpe, "fitness": fitness, "turnover": turnover})
        else:
            # Print the exact API error message
            error_msg = r.get("simulation", {}).get("message", "Unknown error")
            print(f"❌ Simulation failed. Reason: {error_msg}")
            
    except Exception as e:
        print(f"❌ Error: {e}")
    print("-" * 50)
    time.sleep(1)

# 3. Display results
if results_top1000:
    print("\n--- FINAL RESULTS (TOP1000 ANALYST DATA) ---")
    display(pd.DataFrame(results_top1000).sort_values(by="fitness", ascending=False))
else:
    print("\nNo simulations completed successfully.")

Extracted 5 analyst expressions. Simulating on TOP1000...

⏳ Simulating: group_neutralize(rank(ts_zscore(ts_backfill(anl4_adxqfv110_m...
[07:06:14] simulating: group_neutralize(rank(ts_zscore(ts_backfill(anl4_adxqfv110_mean, 5), 6
❌ Simulation failed. Reason: Operator ts_backfill does not support event inputs. <linkToCommonErrorMessages>Learn more</linkToCommonErrorMessages>
--------------------------------------------------
⏳ Simulating: group_neutralize(ts_delta(ts_backfill(subtract(anl4_adxqfv11...
[07:06:28] simulating: group_neutralize(ts_delta(ts_backfill(subtract(anl4_adxqfv110_pu, anl4
❌ Simulation failed. Reason: Operator subtract does not support event inputs. <linkToCommonErrorMessages>Learn more</linkToCommonErrorMessages>
--------------------------------------------------
⏳ Simulating: rank(ts_decay_linear(ts_delta(ts_backfill(anl4_ady_mean, 5),...
[07:07:02] simulating: rank(ts_decay_linear(ts_delta(ts_backfill(anl4_ady_mean, 5), 20), 10))
❌ Simulation failed. Reason: Ope

,expression,sharpe,fitness,turnover
0,rank(reverse(ts_zscore(ts_backfill(divide(adj_...,-0.81,-0.37,0.1469


In [3]:
%pip install tqdm

Note: you may need to restart the kernel to use updated packages.


In [4]:
import ace_lib as ace
import helpful_functions as hf
print("✅ Success! The engines are loaded.")

✅ Success! The engines are loaded.


In [ ]:
import ace_lib as ace
import helpful_functions as hf
import pandas as pd
import nest_asyncio
import os
from openai import OpenAI
from dotenv import load_dotenv
import re

nest_asyncio.apply()
load_dotenv()

print("Logging into WorldQuant BRAIN...")
s = ace.start_session(force_refresh=False)

Logging into WorldQuant BRAIN...


In [5]:
import os
import json
from getpass import getpass

# 1. Create the secrets folder
secrets_dir = os.path.expanduser("~/secrets")
os.makedirs(secrets_dir, exist_ok=True)

# 2. Define your credentials (REPLACE WITH YOUR ACTUAL EMAIL AND PASSWORD)
creds = {
    "email": input("BRAIN email: "),
    "password": getpass("BRAIN password: "),
}

# 3. Save the file
filepath = os.path.join(secrets_dir, "platform-brain.json")
with open(filepath, "w") as f:
    json.dump(creds, f)

print(f"✅ Credentials saved securely to {filepath}")

SyntaxError: unterminated string literal (detected at line 11) (4115704535.py, line 11)